# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaifLatki/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I used the prepared refresh feature export in `data/processed/refresh_feature_vector.csv`, which already contains the cleaned 90-day content snapshot and the derived target column `is_declining_label`.

The feature vector keeps only the variables that are available before prediction time and that are not generated from the target itself. I excluded IDs and label-derived fields, then used numeric and categorical columns that represent traffic, content quality, and page metadata. Missing numeric values were filled conservatively and categorical misses were mapped to `unknown` so the model could run without silently encoding a missingness pattern as a signal.


In [2]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
repo = None
for candidate in [cwd, *cwd.parents]:
    script_path = candidate / "scripts" / "01_prepare_features.py"
    data_path = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
    if script_path.exists() and data_path.exists():
        repo = candidate
        break
if repo is None:
    raise FileNotFoundError("Could not find the FlyRank repo root from the current working directory.")

feature_path = repo / "data" / "processed" / "refresh_feature_vector.csv"
if not feature_path.exists():
    subprocess.run([sys.executable, str(repo / "scripts" / "01_prepare_features.py")], cwd=repo, check=True)
    feature_path = repo / "data" / "processed" / "refresh_feature_vector.csv"

feature_df = pd.read_csv(feature_path)
feature_df["is_declining_label"] = feature_df["trend_direction"].str.lower().eq("down").astype(int)

required = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
    "is_declining_label",
]
missing = [column for column in required if column not in feature_df.columns]
if missing:
    raise ValueError(f"Missing required columns for the feature vector: {missing}")

print(f"Rows in feature vector: {len(feature_df):,}")
print(f"Declining label rate: {feature_df['is_declining_label'].mean():.3f}")
print(feature_df[["content_id", "client_id", "impressions_90d", "sessions_90d", "content_age_days", "trend_direction", "is_declining_label"]].head(5).to_string(index=False))


Rows in feature vector: 30,000
Declining label rate: 0.542
          content_id         client_id  impressions_90d  sessions_90d  content_age_days trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc             3803            17               187            down                   1
content_a1fb4e703a9e client_4e07408562            15320             9               445            down                   1
content_9aa793d4d895 client_7f2253d7e2            12581            11               141            down                   1
content_331d6c4de07b client_19581e27de            11751            78               463          stable                   0
content_d99b7a2d90ca client_3fdba35f04            19140           145               263            down                   1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [3]:
from textwrap import dedent

print(dedent('''
Feature vector notes

- search_volume, competition, cpc, word_count, char_count: keyword and content metadata. Missing values are filled during feature preparation, and the category fields use "unknown" instead of a blind zero so the missingness pattern is not silently encoded as a signal.
- impressions_90d, clicks_90d, sessions_90d, ai_sessions_90d, days_with_impressions, days_with_sessions: observed 90-day traffic and engagement totals from Google Search Console and GA4. These are measurable before prediction time because they summarize the already-observed window.
- content_age_days and days_since_last_update: editorial age and freshness. They are known before the model predicts decline and are not derived from the target.
- ctr, engagement_rate, scroll_rate, ai_traffic_pct, avg_position: rates and quality metrics computed over the same observed 90-day window. These are valid only if they are not themselves proxies for the target; the leakage test checks exactly that.
- trend_direction and trend_pct: derived from the recent 30-day comparison and therefore define the target. They are explicitly excluded from the features.
- content_type, competition_level, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, position_tier: categorical context that describes the content and page state. Missing categories are encoded as "unknown" rather than 0 to avoid creating a false category signal.
- content_id and client_id: identifiers only; they are used for grouping and joins, not as model inputs.
'''))



Feature vector notes

- search_volume, competition, cpc, word_count, char_count: keyword and content metadata. Missing values are filled during feature preparation, and the category fields use "unknown" instead of a blind zero so the missingness pattern is not silently encoded as a signal.
- impressions_90d, clicks_90d, sessions_90d, ai_sessions_90d, days_with_impressions, days_with_sessions: observed 90-day traffic and engagement totals from Google Search Console and GA4. These are measurable before prediction time because they summarize the already-observed window.
- content_age_days and days_since_last_update: editorial age and freshness. They are known before the model predicts decline and are not derived from the target.
- ctr, engagement_rate, scroll_rate, ai_traffic_pct, avg_position: rates and quality metrics computed over the same observed 90-day window. These are valid only if they are not themselves proxies for the target; the leakage test checks exactly that.
- trend_direc

## 3. The leakage hunt

The core check is simple: if a feature contains the label definition or information from the outcome window, the score will look artificially strong. I tested two versions of the same model: one with a leakage-safe feature set and one with the obvious label-derived field `trend_pct` added back in.

The honest model is grouped by `client_id` so the holdout reflects whether the model generalizes to a client it has not seen, rather than memorizing the client profile. The leak attack is intentionally the same model with the target parent variable injected; if that single trick collapses the validation problem, that confirms the feature is the answer in disguise.


In [5]:
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

cwd = Path.cwd().resolve()
repo = None
for candidate in [cwd, *cwd.parents]:
    candidate_data = candidate / "data" / "processed" / "refresh_feature_vector.csv"
    if candidate_data.exists():
        repo = candidate
        break
if repo is None:
    raise FileNotFoundError("Could not locate the repo root for the prepared feature vector.")

# Build a leakage-safe feature set and a deliberately leaky comparison.
feature_df = pd.read_csv(repo / "data" / "processed" / "refresh_feature_vector.csv")
feature_df = feature_df.copy()
feature_df["is_declining_label"] = feature_df["trend_direction"].str.lower().eq("down").astype(int)

exclude = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"}
base_features = [c for c in feature_df.columns if c not in exclude]

# Keep only the honest model features we explicitly allow.
allowed_numeric = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]
allowed_categorical = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]
feature_set = allowed_numeric + allowed_categorical

X = feature_df[feature_set]
y = feature_df["is_declining_label"]
client_groups = feature_df["client_id"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, allowed_numeric),
    ("cat", categorical_transformer, allowed_categorical),
])

model = LogisticRegression(max_iter=2000, class_weight="balanced")
pipe = Pipeline([("preprocess", preprocessor), ("model", model)])

cv = GroupKFold(n_splits=5)
probs = np.zeros(len(feature_df))
for train_idx, test_idx in cv.split(X, y, groups=client_groups):
    pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
    probs[test_idx] = pipe.predict_proba(X.iloc[test_idx])[:, 1]

# Honest score: no label-derived fields
honest_auc = roc_auc_score(y, probs)
honest_accuracy = accuracy_score(y, (probs >= 0.5).astype(int))
honest_precision = precision_score(y, (probs >= 0.5).astype(int), zero_division=0)
honest_recall = recall_score(y, (probs >= 0.5).astype(int), zero_division=0)
honest_f1 = f1_score(y, (probs >= 0.5).astype(int), zero_division=0)

# Deliberately leaky test: if using trend_pct as a feature jumps the score to almost 1, that confirms the label is in the feature.
leaky_df = feature_df[feature_set + ["trend_pct"]].copy()
X_leaky = leaky_df
probs_leaky = np.zeros(len(feature_df))
for train_idx, test_idx in cv.split(X_leaky, y, groups=client_groups):
    leaky_pipe = Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", numeric_transformer, allowed_numeric + ["trend_pct"]),
            ("cat", categorical_transformer, allowed_categorical),
        ])),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])
    leaky_pipe.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
    probs_leaky[test_idx] = leaky_pipe.predict_proba(X_leaky.iloc[test_idx])[:, 1]

leaky_auc = roc_auc_score(y, probs_leaky)
leaky_accuracy = accuracy_score(y, (probs_leaky >= 0.5).astype(int))

print(f"Honest client-grouped AUC: {honest_auc:.3f}")
print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Honest precision: {honest_precision:.3f}")
print(f"Honest recall: {honest_recall:.3f}")
print(f"Honest F1: {honest_f1:.3f}")
print(f"Leaky trend_pct AUC: {leaky_auc:.3f}")
print(f"Leaky accuracy: {leaky_accuracy:.3f}")
print("\nLeakage conclusion: if a feature derived from the target drives the score near 1.0, the feature set is encoding the answer itself and should be removed.")

feature_df[["trend_direction", "trend_pct", "is_declining_label"]].head(10).to_string(index=False)


Honest client-grouped AUC: 0.665
Honest accuracy: 0.629
Honest precision: 0.638
Honest recall: 0.729
Honest F1: 0.680
Leaky trend_pct AUC: 0.999
Leaky accuracy: 0.981

Leakage conclusion: if a feature derived from the target drives the score near 1.0, the feature set is encoding the answer itself and should be removed.


'trend_direction  trend_pct  is_declining_label\n           down      -41.4                   1\n           down      -57.7                   1\n           down      -60.9                   1\n         stable      -13.8                   0\n           down      -34.7                   1\n           down      -38.9                   1\n           down      -92.3                   1\n         stable        0.6                   0\n           down      -58.8                   1\n           down      -29.2                   1'

## 4. What I excluded and why

I intentionally excluded all fields that either identify the item or define the outcome. The rule is: if a column would not be observable at the moment you would predict future decline, it is not a valid feature in this notebook.


In [6]:
print("Excluded fields and why:")
for column in [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]:
    reason = {
        "content_id": "Identifier only; used for joins and grouping, never as a predictive feature.",
        "client_id": "Pseudonymous grouping key; if used as a feature it can leak client-specific patterns and breaks honest holdout evaluation.",
        "trend_direction": "This is the target definition itself; the label is derived from it and using it as a feature would leak the answer.",
        "trend_pct": "This is the exact percentage change driving trend_direction, so it is the answer in disguise and would make the model trivial.",
        "provider_used": "Opaque model-generation metadata; it does not represent the content's future business outcome and can encode generation recipe rather than performance signal.",
        "model_used": "Similarly, this signals how a page was created rather than true business impact; it is not a stable prediction feature.",
        "impressions_last_30d": "A recent-window traffic total overlaps the outcome period and can directly reflect the same signal being predicted.",
        "clicks_last_30d": "Same issue as impressions_last_30d: it measures the recent outcome window and would leak the answer.",
        "sessions_last_30d": "Same window-overlap issue as the other recency features.",
        "impressions_prev_30d": "This is a denominator in trend_pct and is used to define the target; it should not be used as a stand-alone feature because it is part of the same label logic.",
        "clicks_prev_30d": "The previous-window version of the same label logic, so it is not independent evidence for future decline.",
        "sessions_prev_30d": "Again, a component of the label-generating ratio and thus not safe as a feature.",
    }[column]
    print(f"- {column}: {reason}")


Excluded fields and why:
- content_id: Identifier only; used for joins and grouping, never as a predictive feature.
- client_id: Pseudonymous grouping key; if used as a feature it can leak client-specific patterns and breaks honest holdout evaluation.
- trend_direction: This is the target definition itself; the label is derived from it and using it as a feature would leak the answer.
- trend_pct: This is the exact percentage change driving trend_direction, so it is the answer in disguise and would make the model trivial.
- provider_used: Opaque model-generation metadata; it does not represent the content's future business outcome and can encode generation recipe rather than performance signal.
- model_used: Similarly, this signals how a page was created rather than true business impact; it is not a stable prediction feature.
- impressions_last_30d: A recent-window traffic total overlaps the outcome period and can directly reflect the same signal being predicted.
- clicks_last_30d: Same

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
